# 05 — Correlation-aware shrinkage: `graph_horseshoe`

Phase 1 (`simulate_twas_dataset`) gives GReX with real gene-gene correlation (adjacent genes
share cis-eQTLs), but every prior so far (`regularized_horseshoe`, `horseshoe`,
`bayesian_lasso`) shrinks each gene **independently** — none of them actually uses that
correlation. `graph_horseshoe` closes that gap: identical global-local-slab structure to
`regularized_horseshoe`, but the latent draw is correlated via a gene-gene correlation matrix
instead of iid.

This notebook compares it against the baselines twice: once with the **oracle** correlation
(the closed-form population truth, `true_grex_correlation`) and once with a **realistic**
correlation **estimated from the training data** (`estimate_gene_correlation`, Ledoit-Wolf,
leakage-safe per fold via `BayesBuilder(corr_estimate=True)`).

## Setup

In [ ]:
import numpy as np
import ptgs_bc as ptgs
from ptgs_bc import (simulate_twas_dataset, true_grex_correlation, estimate_gene_correlation,
                     ElasticNetBuilder, BayesBuilder, run_benchmark, summary_table,
                     per_fold_table, plot_performance, plot_comparison, plot_corr_recovery,
                     compare_priors, plot_prior_comparison)
%matplotlib inline
print("ptgs_bc", ptgs.__version__)

## Simulate TWAS GReX with pronounced gene-gene correlation

Higher `window_overlap`/`ld_rho` than notebook 01's demo, so neighboring genes share more
cis-SNPs and the correlation structure `graph_horseshoe` is meant to exploit is stronger.

In [ ]:
ds, true_w = simulate_twas_dataset(
    n_samples=180, n_genes=250, n_snps_per_gene=15, window_overlap=10, ld_rho=0.75,
    eqtl_sparsity=0.3, n_causal_genes=8, trait_pve=0.4, effect_sd=1.5, seed=0)
C_true = true_grex_correlation(ds.meta)
print(f"p={ds.n_genes} genes, n={ds.n_samples} samples, 8 causal | "
      f"adjacent-gene true |corr|={np.abs(np.diag(C_true, 1)).mean():.3f}")

## Oracle correlation: `graph_horseshoe` vs. the baselines

Upper bound: `graph_horseshoe` is handed the exact population correlation matrix. If it
can't beat `regularized_horseshoe` even here, the correlation structure isn't the
bottleneck in this regime.

In [ ]:
builders_oracle = [
    ElasticNetBuilder(),
    BayesBuilder(prior="regularized_horseshoe", p0=8, num_warmup=300, num_samples=300),
    BayesBuilder(prior="graph_horseshoe", p0=8, num_warmup=300, num_samples=300, corr=C_true),
]
res_oracle = run_benchmark(builders_oracle, ds, outer_k=5, seed=0)
summary_table(res_oracle)

In [ ]:
plot_comparison(res_oracle)

## Estimated correlation (realistic): does `graph_horseshoe` still help?

`corr_estimate=True` re-estimates the correlation matrix (Ledoit-Wolf) from each fold's
*training* GReX only — no test-set leakage — the version you'd actually run on real data.

In [ ]:
builders_est = [
    ElasticNetBuilder(),
    BayesBuilder(prior="regularized_horseshoe", p0=8, num_warmup=300, num_samples=300),
    BayesBuilder(prior="graph_horseshoe", p0=8, num_warmup=300, num_samples=300,
                 corr_estimate=True),
]
res_est = run_benchmark(builders_est, ds, outer_k=5, seed=0)
summary_table(res_est)

In [ ]:
plot_comparison(res_est)

## How far is the estimate from the oracle?

A diagonal cloud below means the Ledoit-Wolf estimate is close to the closed-form truth; a
flattened one means `graph_horseshoe` is working from a noisy graph in the estimated run above.

In [ ]:
C_est = estimate_gene_correlation(ds.grex.to_numpy())
plot_corr_recovery(C_true, C_est)

## WAIC comparison across priors

Cheaper than a full nested-CV refit: one MCMC fit per prior on the full data, compared by
WAIC. `hyper_overrides` supplies each structured prior's own extra hyperparameter (`corr`
for `graph_horseshoe`) that the shared `p0`/`num_warmup`/... knobs can't express.

In [ ]:
cmp, lls = compare_priors(
    ds, priors=("regularized_horseshoe", "graph_horseshoe"), p0=8,
    num_warmup=300, num_samples=300, seed=0,
    hyper_overrides={"graph_horseshoe": {"corr": C_true}})
cmp

In [ ]:
plot_prior_comparison(cmp)

## Reading the result

With the oracle correlation, `graph_horseshoe` has every advantage the correlation structure
can offer — this is the ceiling. The estimated-correlation run shows how much of that
advantage survives when the graph itself has to be learned from the same small training set
(the realistic case). If the gap between oracle and estimated is large, that's a sign the
correlation-aware prior needs either more training samples or a better-conditioned estimator
than plain Ledoit-Wolf before it's worth the extra modeling complexity over
`regularized_horseshoe` on real data.